In [47]:
!pip install catboost & lightgbm --config-settings=cmake.define.USE_GPU=ON

/bin/bash: line 1: lightgbm: command not found


In [48]:
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold

In [49]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train.columns)

Index(['id', 'age', 'alcohol_consumption_per_week',
       'physical_activity_minutes_per_week', 'diet_score',
       'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi',
       'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate',
       'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol',
       'triglycerides', 'gender', 'ethnicity', 'education_level',
       'income_level', 'smoking_status', 'employment_status',
       'family_history_diabetes', 'hypertension_history',
       'cardiovascular_history', 'diagnosed_diabetes'],
      dtype='object')


In [50]:
def engineer_features(df):
    # คัดลอก DataFrame เพื่อไม่ให้กระทบตัวแปรเดิม
    df = df.copy()

    # --- 1. กลุ่มดัชนีร่างกาย (Body Metrics) ---
    # สัดส่วนเอวต่อสะโพก (Waist-to-Hip Ratio) บ่งบอกไขมันช่องท้อง
    # (มีมาให้แล้ว แต่เราสามารถแบ่งกลุ่มตามความเสี่ยงได้)
    df['is_obese'] = (df['bmi'] >= 30).astype(int)
    df['high_whr'] = df.apply(lambda x: 1 if (x['gender'] == 'Male' and x['waist_to_hip_ratio'] > 0.9)
                              else (1 if x['waist_to_hip_ratio'] > 0.85 else 0), axis=1)

    # --- 2. กลุ่มความดันโลหิต (Blood Pressure) ---
    # Pulse Pressure: ค่าความต่างของความดันตัวบนและตัวล่าง
    df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']
    # Mean Arterial Pressure (MAP): ความดันเฉลี่ยในหลอดเลือด
    df['map'] = (df['systolic_bp'] + 2 * df['diastolic_bp']) / 3
    # Hypertension Stage (เกณฑ์ความดันสูง)
    df['is_hypertensive'] = ((df['systolic_bp'] >= 140) | (df['diastolic_bp'] >= 90)).astype(int)

    # --- 3. กลุ่มไขมันในเลือด (Lipid Profile) ---
    # Total Cholesterol / HDL Ratio (ยิ่งสูงยิ่งเสี่ยง)
    df['chol_hdl_ratio'] = df['cholesterol_total'] / (df['hdl_cholesterol'] + 1e-9)
    # LDL / HDL Ratio
    df['ldl_hdl_ratio'] = df['ldl_cholesterol'] / (df['hdl_cholesterol'] + 1e-9)
    # Triglyceride / HDL Ratio (ตัวบ่งชี้ภาวะดื้ออินซูลินที่แม่นยำมาก)
    df['tg_hdl_ratio'] = df['triglycerides'] / (df['hdl_cholesterol'] + 1e-9)
    # Non-HDL Cholesterol
    df['non_hdl_cholesterol'] = df['cholesterol_total'] - df['hdl_cholesterol']

    # --- 4. กลุ่มไลฟ์สไตล์ (Lifestyle Interactions) ---
    # อัตราส่วนกิจกรรมต่อการจ้องจอ (Active vs Sedentary)
    df['activity_screen_ratio'] = df['physical_activity_minutes_per_week'] / (df['screen_time_hours_per_day'] * 7 + 1)
    # คะแนนสุขภาพรวม (Healthy Score)
    # (ยิ่งกินดี นอนพอ ออกกำลังกายเยอะ คะแนนยิ่งสูง)
    df['lifestyle_score'] = (df['diet_score'] + (df['sleep_hours_per_day'] * 2)) - (df['alcohol_consumption_per_week'] / 7)

    # --- 5. กลุ่มประวัติครอบครัวและโรคประจำตัว (Risk Factors) ---
    # นับรวมความเสี่ยงทางกรรมพันธุ์และโรคเดิม
    df['total_risk_histories'] = df[['family_history_diabetes', 'hypertension_history', 'cardiovascular_history']].sum(axis=1)

    # --- 6. การจัดการข้อมูลที่เป็นกลุ่ม (Encoding) ---
    # แปลงระดับการศึกษาและรายได้เป็นตัวเลข (Ordinal Encoding)
    # สมมติลำดับจากน้อยไปมาก (ปรับตามข้อมูลจริงของคุณ)
    edu_map = {'High School': 1, 'Associate Degree': 2, 'Bachelor': 3, 'Master': 4, 'PhD': 5}
    income_map = {'Low': 1, 'Medium': 2, 'High': 3}

    if 'education_level' in df.columns:
        df['education_level_coded'] = df['education_level'].map(edu_map).fillna(0)
    if 'income_level' in df.columns:
        df['income_level_coded'] = df['income_level'].map(income_map).fillna(0)

    # --- 7. ลบ Column ที่ไม่จำเป็น ---
    # 'id' ไม่ควรนำมาเทรน
    if 'id' in df.columns:
        df = df.drop(columns=['id'])

    return df

In [51]:
train_fe = engineer_features(train)
test_fe = engineer_features(test)

X = train_fe.drop(columns='diagnosed_diabetes')
y = train_fe['diagnosed_diabetes']

In [52]:
cat_features = ['gender', 'ethnicity', 'smoking_status', 'employment_status']
# ตัวแปรที่มีลำดับ (Ordinal)
ord_features = ['education_level', 'income_level']
# ตัวแปรตัวเลข
num_features = [col for col in train_fe.columns if col not in cat_features + ord_features + ['diagnosed_diabetes']]

In [53]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('ord', OrdinalEncoder(), ord_features)
    ])

In [54]:
base_models = [
    ('xgb', XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        tree_method='hist',
        device='cuda'
    )),

    ('lgb', LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        device='gpu'
    )),

    ('cat', CatBoostClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        depth=6,
        verbose=0,
        task_type="GPU"
    ))
]

In [55]:
stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(),
    cv=5,
    stack_method='predict_proba',
    n_jobs=1
)

In [56]:
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', stacking_model)
])

In [57]:
clf.fit(X, y)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:43:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Number of positive: 436307, number of negative: 263693
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3322
[LightGBM] [Info] Number of data points in the train set: 700000, number of used features: 48
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 28 dense feature groups (18.69 MB) transferred to GPU in 0.028154 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.623296 -> initscore=0.503561
[LightGBM] [Info] Start training from score 0.503561


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:44:40] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [18:44:47] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:44:49] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, f

[LightGBM] [Info] Number of positive: 349045, number of negative: 210955
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3323
[LightGBM] [Info] Number of data points in the train set: 560000, number of used features: 48
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 28 dense feature groups (14.95 MB) transferred to GPU in 0.021663 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.623295 -> initscore=0.503556
[LightGBM] [Info] Start training from score 0.503556


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

[LightGBM] [Info] Number of positive: 349046, number of negative: 210954
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3323
[LightGBM] [Info] Number of data points in the train set: 560000, number of used features: 48
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 28 dense feature groups (14.95 MB) transferred to GPU in 0.021841 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.623296 -> initscore=0.503564
[LightGBM] [Info] Start training from score 0.503564


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 349046, number of negative: 210954
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3317
[LightGBM] [Info] Number of data points in the train set: 560000, number of used features: 48
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 28 dense feature groups (14.95 MB) transferred to GPU in 0.022978 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.623296 -> initscore=0.503564
[LightGBM] [Info] Start training from score 0.503564


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['age',
                                                   'alcohol_consumption_per_week',
                                                   'physical_activity_minutes_per_week',
                                                   'diet_score',
                                                   'sleep_hours_per_day',
                                                   'screen_time_hours_per_day',
                                                   'bmi', 'waist_to_hip_ratio',
                                                   'systolic_bp',
                                                   'diastolic_bp', 'heart_rate',
                                                   'cholesterol_total',
                                                   'hdl_cholesterol',
                                                   'ldl_cholesterol',
                                                   't...
                                                               monotone_constraints=None,
                                                               multi_strategy=None,
                                                               n_estimators=1000,
                                                               n_jobs=None,
                                                               num_parallel_tree=None, ...)),
                                                ('lgb',
                                                 LGBMClassifier(device='gpu',
                                                                learning_rate=0.05,
                                                                n_estimators=1000,
                                                                random_state=42)),
                                                ('cat',
                                                 CatBoostClassifier(depth=6, learning_rate=0.05, n_estimators=1000, task_type='GPU', verbose=0))],
                                    final_estimator=LogisticRegression(),
                                    n_jobs=1, stack_method='predict_proba'))])

In [58]:
probs = clf.predict_proba(test_fe)[:, 1]

print(probs)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[0.45428088 0.68184327 0.81225214 ... 0.575848   0.61264469 0.60575073]


In [60]:
submission = pd.DataFrame({
    "id": test['id'],
    "diagnosed_diabetes": probs
})

submission.to_csv("submission.csv", index=False)